# Rythu Mitra: IndicBERT / MuRIL fine-tuning (not executed in the project environment)
Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. Upload the completed project ZIP. This trains only `data/train.csv`, selects by the existing grouped validation split, and evaluates the untouched workbook test and both authored challenge sets. No benchmark query is used for fitting. IndicBERT may require accepting the model owner's conditions; authenticate through the notebook only if needed. If unavailable, MuRIL is tried. No metric is claimed until these cells actually finish.
Sources: https://huggingface.co/ai4bharat/indic-bert , https://huggingface.co/google/muril-base-cased , https://huggingface.co/docs/transformers/v4.46.3/en/main_classes/trainer


In [ ]:
%pip install -q "transformers==4.46.3" "datasets>=3,<4" "accelerate>=1,<2" sentencepiece scikit-learn pandas openpyxl matplotlib


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, json, sys
uploaded = files.upload()
archive = next(Path(name) for name in uploaded if name.endswith('.zip'))
ROOT = Path('/content/rythu_mitra').resolve()
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (ROOT/name).resolve().is_relative_to(ROOT):
            raise ValueError('Unsafe archive path')
    z.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


In [ ]:
import numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
set_seed(42)
train = pd.read_csv('data/train.csv')
val = pd.read_csv('data/validation_original.csv')
test = pd.read_csv('data/test_original.csv')
hand1 = pd.read_csv('data/handwritten_test.csv')
hand2 = pd.read_csv('data/handwritten_test_v2.csv')
assert not set(train.normalized) & (set(val.normalized) | set(test.normalized))
assert not set(train.group) & (set(val.group) | set(test.group))
from ml.preprocess import normalize
for challenge in [hand1, hand2]:
    assert not set(challenge['query'].map(normalize)) & set(train.normalized)
labels = sorted(train.intent.unique())
label2id = {v:i for i,v in enumerate(labels)}
print({'cuda_available': torch.cuda.is_available(), 'training_rows':len(train), 'validation_rows':len(val), 'test_rows':len(test)})
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime before fine-tuning. No fake CPU result is reported.')


In [ ]:
errors = {}
for checkpoint in ['ai4bharat/indic-bert', 'google/muril-base-cased']:
    try:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=len(labels),
                 id2label=dict(enumerate(labels)), label2id=label2id)
        break
    except Exception as error:
        errors[checkpoint] = type(error).__name__
else:
    raise RuntimeError('Neither pretrained checkpoint could be loaded: '+str(errors))
print('Actually loaded checkpoint:',checkpoint)
def dataset(frame):
    d = Dataset.from_dict({'text':frame['query'].astype(str).tolist(), 'labels':[label2id[x] for x in frame.intent]})
    return d.map(lambda batch:tokenizer(batch['text'], truncation=True, max_length=128), batched=True).remove_columns('text')
def compute_metrics(result):
    y = result.label_ids
    pred = np.argmax(result.predictions, axis=-1)
    values = {'accuracy':float(accuracy_score(y,pred))}
    for average in ['macro','weighted']:
        p,r,f,_ = precision_recall_fscore_support(y,pred,average=average,zero_division=0)
        values.update({average+'_precision':float(p),average+'_recall':float(r),average+'_f1':float(f)})
    return values
args = TrainingArguments(output_dir='neural_run', num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='macro_f1', greater_is_better=True, save_total_limit=2,
    seed=42, data_seed=42, report_to='none', fp16=True)
trainer = Trainer(model=model,args=args,train_dataset=dataset(train),eval_dataset=dataset(val),
    tokenizer=tokenizer,data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=compute_metrics)
trainer.train()


In [ ]:
results = {'checkpoint':checkpoint,'training_rows':len(train),'epochs':3,'seed':42,'status':'actually_run'}
for name,frame in [('workbook_test',test),('handwritten_v1',hand1),('primary_handwritten_v2',hand2)]:
    output = trainer.predict(dataset(frame))
    metric = compute_metrics(output)
    pred = np.argmax(output.predictions,axis=-1)
    metric['per_class'] = classification_report([label2id[x] for x in frame.intent], pred,
                         labels=list(range(len(labels))),target_names=labels,output_dict=True,zero_division=0)
    results[name] = metric
    fig,ax=plt.subplots(figsize=(10,8))
    ConfusionMatrixDisplay.from_predictions([label2id[x] for x in frame.intent],pred,
        labels=list(range(len(labels))),display_labels=labels,ax=ax,xticks_rotation=45,colorbar=False)
    fig.tight_layout();fig.savefig('neural_run/'+name+'_confusion.png');plt.close(fig)
    print(name, {k:v for k,v in metric.items() if k!='per_class'})
Path('neural_run/metrics.json').write_text(json.dumps(results,indent=2))
trainer.save_model('neural_run/best_model')
tokenizer.save_pretrained('neural_run/best_model')
import shutil
shutil.make_archive('/content/rythu_mitra_neural_results','zip',ROOT/'neural_run')
files.download('/content/rythu_mitra_neural_results.zip')


The primary score is the newly authored handwritten-v2 result, not the easier templated workbook score. Retain the run metadata, per-class reports and plots. Update the project comparison only after this run has completed. These are research topic classifiers, not diagnosis tools; no crop treatment is generated from the neural model.
